# Ejemplo de entrenamiento con FLAML para forecasting
Genera un modelo de ejemplo, requirements.txt y esquema de datos para integrarse con el pipeline de inferencia en Azure.

In [ ]:
import os
from datetime import datetime, timedelta
import numpy as np
import pandas as pd
from flaml import AutoML
import joblib

MODEL_NAME = "demo-model"
OUTPUT_DIR = f"artifacts/{MODEL_NAME}"
os.makedirs(f"{OUTPUT_DIR}/model", exist_ok=True)
os.makedirs(f"{OUTPUT_DIR}/requirements", exist_ok=True)
os.makedirs(f"{OUTPUT_DIR}/data", exist_ok=True)


In [ ]:
# Crear un dataset sintético diario
n_points = 120
dates = pd.date_range(end=datetime.utcnow(), periods=n_points, freq="D")
signal = np.sin(np.linspace(0, 6, n_points)) + np.random.normal(0, 0.2, n_points)
df = pd.DataFrame({"ds": dates, "y": signal})
df["dayofweek"] = df.ds.dt.dayofweek
df.tail()

In [ ]:
automl = AutoML()
settings = {
    'time_budget': 30,
    'task': 'ts_forecast',
    'metric': 'mape',
    'period': 7,
    'max_iter': 20,
    'time_col': 'ds',
    'target_col': 'y',
    'eval_method': 'holdout',
    'split_ratio': 0.8,
}
automl.fit(dataframe=df, **settings)
print("Mejor estimador", automl.best_estimator)


In [ ]:
# Guardar modelo y artefactos
model_path = f"{OUTPUT_DIR}/model/{MODEL_NAME}.pickle"
joblib.dump(automl, model_path)
print(f"Modelo guardado en {model_path}")

requirements = [
    'flaml==2.1.2',
    'pandas==2.2.2',
    'numpy==1.26.4',
    'scikit-learn==1.4.2',
]
with open(f"{OUTPUT_DIR}/requirements/requirements.txt", 'w', encoding='utf-8') as f:
    f.write('
'.join(requirements))

# Esquema vacío
schema = df.head(0)
schema.to_parquet(f"{OUTPUT_DIR}/data/schema.empty.parquet")
print('Artefactos listos en', OUTPUT_DIR)
